# Notebook 6 — Consolidation: recap, stack check, and exercises

The final notebook ties the series together: a single stack check, a review of how the engines reconcile, suggested exercises, and pointers to the deeper documentation.


## Recap

| Notebook | Covered |
|----------|---------|
| 0 · setup | Environment, install, data |
| 1 · concept | Tidal physics, ½ρU³, turbine mechanics, hotspots |
| 2 · data | Inputs, config, six canonical outputs |
| 3 · general-workflow | Screening → TELEMAC → web pipeline |
| 4 · model | Grid, solver, forcing, outputs |
| 5 · web | Flask API, map, turbines |
| 6 · consolidation | This notebook |


## 6.1 Final stack check

Run one cell that verifies the whole stack: Python, the `model` package, `web.turbines`, the configuration, and the screening output.


In [ ]:
import json
from pathlib import Path

report = {}

try:
    import numpy as np
    report['numpy'] = np.__version__
except ImportError:
    report['numpy'] = 'missing'

try:
    from model.grid import StructuredGrid
    report['model pkg'] = 'ok'
except ImportError:
    report['model pkg'] = 'missing (PYTHONPATH=src)'

try:
    from web.turbines import all_turbine_specs
    report['web.turbines'] = f'{len(all_turbine_specs())} turbines'
except ImportError:
    report['web.turbines'] = 'missing'

try:
    from model.config import load_config
    report['engine'] = load_config()['engine']['name']
except Exception as exc:
    report['engine'] = f'error ({exc})'

root = next((c for c in (Path('.'), Path('../..')) if (c / 'src').is_dir()),
            Path('.'))
report['power raster'] = (
    'present' if (root / 'output' / 'tidal_power_density.tif').exists()
    else 'not found')

print(json.dumps(report, indent=2))


## 6.2 How the engines reconcile

The Python screening is the **nationwide view**; each TELEMAC-2D run is the zoomed-in view of a strait. The two are kept consistent by treating TELEMAC as a nested child of the screening run:

- **Boundary:** sample the parent's own η at the refinement's liquid points (one-way nesting).
- **Bathymetry:** inherit the parent depths/mask at mesh resolution (`bathymetry_source: parent`).
- **Friction:** harmonise Chezy with the screening drag (`friction_coefficient ≈ sqrt(g / cd)`).
- **Comparison:** `reconciliation.json` records max power/speed, p95 and median, and their parent ratios, with acceptance windows.

Full detail: `../engines/RECONCILIATION.md` and `../engines/TELEMAC.md`.


## 6.3 Exercises

Try these on your own:

1. **Resolution sensitivity** — rerun Notebook 4 with `dx = 1000 m` and compare the max speed. Does halving the cell size change the result as expected?
2. **Friction sensitivity** — rerun with `cd = 0.002` and `cd = 0.004`. Which runs faster/slower, and why?
3. **Constituent set** — add S2 to a synthetic boundary and check the spring–neap envelope appears in the sampled mean.
4. **API** — with the web service running, POST a polygon to `/api/area_stats` and compare area/MW/AEP with `/api/resource`.
5. **Turbines** — for the site series in Notebook 5, find which turbine gives the highest capacity factor and explain why.


## 6.4 Self-check

<details>
<summary>What makes a strait a hotspot?</summary>

A narrow, deep channel connecting basins with different tidal phases concentrates the flow (mass conservation), and power scales as the cube of speed.
</details>

<details>
<summary>Why is the primary map layer a *time-mean* power density?</summary>

Energy yield depends on the full spring–neap envelope; a single instant or peak over-states the resource. Averaging over ≥ 15 days is a better proxy for annual energy.
</details>

<details>
<summary>Why can the same web app serve screening and TELEMAC outputs?</summary>

Both engines write the same canonical files (`results.nc`, the four GeoTIFFs, `hotspots.geojson`), so the API is engine-agnostic.
</details>


## 6.5 Further reading

- `../README.md` — documentation index.
- `../concepts/MODEL.md` — full physics and methodology.
- `../architecture/ARCHITECTURE.md` — integrated technical guide.
- `../engines/TELEMAC.md`, `../engines/CASE_AUTHORING.md`, `../engines/POSTPROCESSING.md`, `../engines/RECONCILIATION.md` — refinement backend.
- `../operations/TROUBLESHOOTING.md` — operational fixes.
- `../notebooks/EXPLAINER.ipynb` and `../notebooks/workshop.ipynb` — alternative walkthroughs.
- `../../src/README.md` — dataset acquisition and step-by-step setup.


---

[Index](README.md) · [← 5.web.ipynb](5.web.ipynb)
